In [ ]:
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class DenseFeatureLayer(nn.Module):
    def __init__(
        self,
        input_size: int,
        nrof_cat: dict[str, int],
        emb_dim: int,
        emb_columns: list[str],
        numeric_columns: list[str],
    ) -> None:
        super().__init__()

        self.emb_columns = emb_columns
        self.numeric_columns = numeric_columns

        self.embeddings = nn.ModuleDict()
        for col in self.emb_columns:
            self.embeddings[col] = nn.Embedding(nrof_cat[col], emb_dim)

        self.feature_bn = nn.BatchNorm1d(input_size)

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor) -> Tensor:
        numeric_feats = torch.as_tensor(pd.DataFrame(x)[self.numeric_columns].to_numpy(), dtype=torch.float)

        embs = [
            self.embeddings[col](torch.tensor(x[col], dtype=torch.long))
            for col in self.emb_columns
        ]
        emb_output = torch.cat(embs, dim=1)

        concat_input = torch.cat([numeric_feats, emb_output], dim=1)
        output = self.feature_bn(concat_input)

        return output

In [ ]:
class GLULayer(nn.Module):
    def __init__(self, input_size: int, output_size: int) -> None:
        super().__init__()

        self.fc = nn.Linear(input_size, output_size)
        self.fc_bn = nn.BatchNorm1d(output_size)

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor) -> Tensor:
        h = self.fc(x)
        h = self.fc_bn(h)
        h = F.glu(h, dim=-1)
        return h

In [ ]:
class FeatureTransformer(nn.Module):
    def __init__(self, n_glu_layers: int, input_size: list[int], output_size: int) -> None:
        super().__init__()

        self.glu_layers = nn.ModuleList(
            [GLULayer(input_size[i], output_size) for i in range(n_glu_layers)]
        )
        self.scale = torch.sqrt(torch.FloatTensor([0.5]))

    def forward(self, x: Tensor) -> Tensor:
        h = x
        for glu_layer in self.glu_layers:
            h = (h + glu_layer(h)) * self.scale
        return ...

In [ ]:
class AttentiveTransformer(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor) -> Tensor:
        ...
        return output

In [ ]:
class Sparsemax(nn.Module):
    def __init__(self, dim: int | None = None) -> None:
        """
        Args:
            dim: The dimension over which to apply the sparsemax function.
        """
        super().__init__()

        self.dim = dim if dim is not None else -1

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Input tensor. First dimension should be the batch size.
        Returns:
            Output tensor of shape [batch_size x n_logits].
        """
        # Sparsemax currently only handles 2-dim tensors,
        # so we reshape to a convenient shape and reshape back after sparsemax
        h = x.transpose(0, self.dim)
        original_size = h.size()
        h = h.reshape(h.size(0), -1)
        h = h.transpose(0, 1)
        dim = 1

        n_logits = h.size(dim)

        # Translate h by max for numerical stability
        h = h - torch.max(h, dim=dim, keepdim=True).values.expand_as(h)

        # Sort h in descending order.
        zs = torch.sort(h, dim=dim, descending=True).values
        range = torch.arange(start=1, end=n_logits + 1, step=1, device=DEVICE, dtype=h.dtype).view(1, -1)
        range = range.expand_as(zs)

        # Determine sparsity of projection
        bound = 1 + range * zs
        cumulative_sum_zs = torch.cumsum(zs, dim)
        is_gt = torch.gt(bound, cumulative_sum_zs).type(h.dtype)
        k = torch.max(is_gt * range, dim, keepdim=True).values

        # Compute threshold function
        zs_sparse = is_gt * zs

        # Compute taus
        taus = (torch.sum(zs_sparse, dim, keepdim=True) - 1) / k
        taus = taus.expand_as(h)

        # Sparsemax
        self.output = torch.max(torch.zeros_like(h), h - taus)

        # Reshape back to original shape
        output = self.output
        output = output.transpose(0, 1)
        output = output.reshape(original_size)
        output = output.transpose(0, self.dim)
        return output

    def backward(self, grad_output):
        dim = 1
        nonzeros = torch.ne(self.output, 0)
        amount = torch.sum(grad_output * nonzeros, dim=dim) / torch.sum(nonzeros, dim=dim)
        self.grad_input = nonzeros * (grad_output - amount.expand_as(grad_output))
        return self.grad_input

In [ ]:
class TabNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor) -> Tensor:
        ...
        return ...